# 02. LangChain Models

LangChain wraps every model provider (OpenAI, Anthropic, Google, Hugging Face and more) in a common interface. There are two families of models:

| Family | Input | Output | Used for |
|---|---|---|---|
| **1. Language models** | Text (a prompt, or a list of messages) | Text | Generating, answering, chatting |
| **2. Embedding models** | Text | A vector of numbers | Semantic search, RAG, clustering, recommendations |

Language models are further split into **LLMs** and **chat models**.

> 🔑 Keys are read from Colab secrets or a local `.env` file (see the README).

In [ ]:
!pip install -q langchain langchain-openai langchain-anthropic langchain-google-genai langchain-huggingface python-dotenv

Check the installed versions (these notes were run with the versions shown):

In [ ]:
import langchain
langchain.__version__

'1.3.9'

In [ ]:
import langchain_anthropic
langchain_anthropic.__version__

'1.4.8'

In [ ]:
import langchain_openai
langchain_openai.__version__

'1.3.3'

In [ ]:
import os

def get_key(name):
    """Read an API key from Colab secrets, or from an environment variable / .env file elsewhere."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ.get(name)

## 1. Language Models

### 1.1 LLMs (Base Models)

- **What they are:** foundation models that take **one string** in and return **one string** out. They are built for single-turn, completion-style tasks, with no conversation history.
- **How they work:** you give a prompt (a question, an instruction or a sentence to continue) and the model generates a continuation.
- **Use cases:** text generation (articles, marketing copy, code), summarization, question answering, translation and information extraction.

In [ ]:
from langchain_openai import OpenAI

# OpenAI() is the completion-style interface, so it needs a completion model
llm = OpenAI(model="gpt-3.5-turbo-instruct", temperature=0.7, api_key=get_key("OPENAI_API_KEY"))

result = llm.invoke("Explain quantum entanglement in one simple sentence.")
print(result)

> 💡 **No live data.** A plain LLM can't answer "What is the weather in Florida today?": it only knows its training data. Live information needs a **tool** (see Agents in notebook 01).

**Anthropic** offers only chat-style models, so with Claude you always use `ChatAnthropic` (next section). Most providers now focus on chat models; plain LLM classes are kept mainly for older completion models.

### 1.2 Chat Models

- **What they are:** models built for **multi-turn conversations**. They take a **list of messages** (the conversation so far) and return one message.
- **How they work:** each message has a role (`SystemMessage` for instructions, `HumanMessage` for the user, `AIMessage` for the model's replies), so the model can follow the dialogue.
- **Use cases:** chatbots and virtual assistants, interactive Q&A and tutoring, role-play simulations and personalized recommendations.

### LLMs vs Chat Models

| Feature | LLMs (base models) | Chat models (instruction-tuned) |
|---|---|---|
| **Purpose** | Free-form text generation | Multi-turn conversations |
| **Training data** | General text (books, articles) | Further fine-tuned on dialogues between users and assistants |
| **Memory and context** | No built-in conversation structure | Takes the full conversation history as a list of messages |
| **Role awareness** | No idea of "user" and "assistant" | Understands system, user and assistant roles |
| **Example models** | GPT-3, Llama 2 7B, Mistral 7B, OPT-1.3B | GPT-4, GPT-4o, Llama 2 Chat, Mistral Instruct, Claude |
| **Use cases** | Text generation, summarization, translation, creative writing, code | Chatbots, virtual assistants, customer support, AI tutors |

### OpenAI Chat Model

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7, api_key=get_key("OPENAI_API_KEY"))

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hi, what's the capital of France?"),
]
response1 = chat_model.invoke(messages)
print(f"AI: {response1.content}")

# Keep the conversation going: add the AI's answer, then the next question
messages.append(response1)
messages.append(HumanMessage(content="And what is its population?"))
response2 = chat_model.invoke(messages)
print(f"AI: {response2.content}")

### Anthropic (Claude) Chat Model

`temperature` controls creativity: `0` gives focused, repeatable answers, and values near `1` give more varied ones.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

anthropic_api_key = get_key("ANTHROPIC_API_KEY")
chat_model = ChatAnthropic(model_name="claude-sonnet-4-6", api_key=anthropic_api_key, temperature=0.7)

# Start a conversation
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hi, what's the capital of USA?")
]
response1 = chat_model.invoke(messages)
print(f"AI: {response1.content}")

AI: Hi! The capital of the United States of America is **Washington, D.C.** (District of Columbia). It has been the nation's capital since 1800 and is home to major government institutions like the White House, the Capitol Building, and the Supreme Court.


The follow-up question works because the whole history is sent again. This is the simplest form of **memory**:

In [ ]:
# Continue the conversation by appending the AI's response and a new user message
messages.append(response1) # Add AI's response to history
messages.append(HumanMessage(content="How many states does it have?"))
response2 = chat_model.invoke(messages)
print(f"AI: {response2.content}")

AI: The United States has **50 states**. Here are a few key facts:

- **The original 13 states** were the founding colonies that declared independence from Britain in 1776.
- **The 49th state** was **Alaska**, admitted in January 1959.
- **The 50th state** was **Hawaii**, admitted in August 1959.

Additionally, the U.S. also has several territories, such as **Puerto Rico**, **Guam**, and the **U.S. Virgin Islands**, but these are not states.


> 💡 The question in the recorded run was typed as "How may states it has?"; it is corrected here, and the answer is the recorded one.

The conversation history now holds every message. Each `AIMessage` also carries metadata such as the model name and **token usage**, which is what you are billed for:

In [ ]:
messages.append(response2)

for m in messages:
    print(f"{type(m).__name__:15} {m.content[:70]}")

print("\nTokens used by the first reply:", response1.usage_metadata)

### Google Gemini Chat Model

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

chat_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=get_key("GOOGLE_API_KEY"), temperature=0.7)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hi, what's the capital of USA?"),
]
response1 = chat_model.invoke(messages)
print(f"AI: {response1.content}")

### Open-Source Chat Models (Hugging Face)

Open-source models such as **Qwen**, **Llama** and **Mistral** can run on Hugging Face's servers (as below, with a free Hugging Face token) or on your own machine. `HuggingFaceEndpoint` connects to the hosted model, and `ChatHuggingFace` gives it the standard chat interface.

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

hugging_face_api_key = get_key("HF_TOKEN")

In [ ]:
Opensource_llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    temperature=0.2,
    huggingfacehub_api_token=hugging_face_api_key,
)

messages = [
    SystemMessage(content="You are a helpful Finance assistant."),
    HumanMessage(content="Hi, what's the best book to learn Finance AI agents?")
]

Open_Source_Model = ChatHuggingFace(llm=Opensource_llm)

response1 = Open_Source_Model.invoke(messages)
print(f"AI: {response1.content}")

messages.append(response1) # Add AI's response to history
messages.append(HumanMessage(content="I need it for new person who does know AI?"))
response2 = Open_Source_Model.invoke(messages)
print(f"AI: {response2.content}")

AI: Learning about Finance AI agents can be a rewarding endeavor, and there are several books that can help you get started. Here are some of the best books that cover various aspects of AI in finance:

1. **"Artificial Intelligence in Finance" by Matthew F. Dixon, Thomas P. Howells, and Prashant V. Mehrotra**
   - This book provides a comprehensive overview of how AI is transforming the finance industry. It covers topics such as machine learning, natural language processing, and reinforcement learning, with practical examples and case studies.

2. **"Machine Learning for Algorithmic Trading: Predictive models to extract signals from market and alternative data for systematic trading with Python" by Stefan Jansen**
   - This book is a practical guide to using machine learning for algorithmic trading. It covers data preprocessing, feature engineering, model selection, and backtesting, with Python code examples.

3. **"Advances in Financial Machine Learning" by Marcos López de Prado**
  

> ⚠️ LLMs can state facts confidently that are wrong, including book titles and authors. Always verify recommendations like these before passing them on.

---

## 2. Embedding Models

- **What they are:** models that turn text (a word, a sentence or a whole document) into a **dense vector of numbers** (an embedding) that captures its **meaning**.
- **How they work:** texts with similar meanings get vectors that are close together in this high-dimensional space, even when they share no words.
- **Use cases:**
  - **Semantic search and RAG:** embed the question, find the document chunks with the closest vectors, and give them to the LLM as context.
  - **Clustering and topic modelling:** group similar feedback, articles or tickets.
  - **Recommendations:** suggest content similar to what a user liked.
  - **Anomaly detection:** spot texts that don't fit the usual pattern.
  - **Deduplication:** find near-identical documents.

### OpenAI Embeddings

`embed_query()` embeds one text. `dimensions=100` shortens the vector (the model's full size is 3,072).

In [ ]:
from langchain_openai import OpenAIEmbeddings

openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=100, api_key=get_key("OPENAI_API_KEY"))

embedding1 = openai_embeddings.embed_query("The quick brown fox jumps over the lazy dog.")
print(f"{embedding1[:5]} ...   ({len(embedding1)} values)")

Recorded output (first 5 of 100 values):
[-0.0519, 0.0396, -0.0468, 0.0848, 0.0055, ...]   (100 values)


`embed_documents()` embeds a list of texts in one call and returns one vector per text:

In [ ]:
documents = [
    "The quick brown fox jumps over the lazy dog.",
    "A fast brown canine leaps over a sleepy hound.",
    "A completely unrelated statement about astrophysics.",
]

result = openai_embeddings.embed_documents(documents)
for vector in result:
    print(f"{vector[:5]} ...   ({len(vector)} values)")

Recorded output (first 5 values of each vector):
[-0.0519, 0.0396, -0.0468, 0.0848, 0.0055, ...]   (100 values)
[-0.1074, -0.0731, -0.0504, -0.0259, 0.0513, ...]   (100 values)
[0.1703, -0.0668, -0.0912, 0.1091, 0.0742, ...]   (100 values)


### Open-Source Embeddings (Hugging Face)

`all-MiniLM-L6-v2` is a small, free model that runs on your own machine (about 90 MB, downloaded on first use) and produces 384-value vectors.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

hf_embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

result = hf_embeddings.embed_documents(documents)
for vector in result:
    print(f"{vector[:5]} ...   ({len(vector)} values)")

Recorded output (first 5 values of each vector):
[0.0439, 0.0589, 0.0482, 0.0775, 0.0267, ...]   (384 values)
[0.0282, 0.0267, 0.0795, 0.0850, 0.0212, ...]   (384 values)
[-0.0101, -0.0331, 0.0085, 0.1031, 0.0110, ...]   (384 values)


### Semantic Search over a PDF

This puts the pieces together into a mini search engine, the core of RAG:

1. **Load** a PDF with `PyPDFLoader`.
2. **Split** it into chunks with `RecursiveCharacterTextSplitter`.
3. **Embed** every chunk and the question.
4. **Compare** them with **cosine similarity** (1 means identical meaning, 0 means unrelated).
5. **Return** the best-matching chunks with their page numbers.

Use any PDF you like, for example a company handbook, a research paper or your lecture notes, and ask a question about it.

In [ ]:
!pip install -q langchain-community langchain-text-splitters pypdf scikit-learn

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity

embedding = OpenAIEmbeddings(model="text-embedding-3-large", api_key=get_key("OPENAI_API_KEY"))

# 1. Load the PDF (upload it to Colab first, or give a local path)
loader = PyPDFLoader("your_document.pdf")
pages = loader.load()

# 2. Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=20)
chunks = splitter.split_documents(pages)
documents = [c.page_content for c in chunks]

# 3. A question about the document
query = "What is the company's employee relations philosophy?"

# 4. Embed the chunks and the question
doc_embeddings = embedding.embed_documents(documents)
query_embedding = embedding.embed_query(query)

# 5. Score every chunk against the question and show the top 3
scores = cosine_similarity([query_embedding], doc_embeddings)[0]
ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

print(f"Query: {query}\n")
for rank, (index, score) in enumerate(ranked[:3], start=1):
    page = chunks[index].metadata.get("page", "?")
    print(f"--- Match {rank} | Score: {score:.4f} | Page: {page} ---")
    print(documents[index][:500])
    print()

The output lists the three most relevant chunks, best first, each with its similarity score and page number. In a full RAG app, those chunks are passed to a chat model together with the question, so it answers from **your** document instead of from memory.